# Advanced Preprocessing

## 🎯 Objective

Advanced preprocessing techniques help prepare complex datasets before training Machine Learning models.

In this notebook, we will learn how to:

- Handle imbalanced datasets
- Apply SMOTE (Oversampling)
- Perform Random Undersampling
- Use Class Weights
- Preprocess Time Series data
- Create Lag Features
- Create Rolling Window Features

These techniques improve model performance on real-world datasets.

# 1. Handling Imbalanced Data

Real-world datasets are often imbalanced, meaning one class contains significantly more samples than another.

For example:

- Fraud Detection
- Disease Prediction
- Spam Detection

If a model is trained on imbalanced data, it usually predicts the majority class and ignores the minority class.

Common techniques include:

- Oversampling (SMOTE)
- Undersampling
- Class Weights

## 1.1 Oversampling (SMOTE)

### What is SMOTE?

SMOTE (Synthetic Minority Over-sampling Technique) creates synthetic samples for the minority class instead of duplicating existing samples.

### Why use SMOTE?

- Balances the dataset
- Reduces model bias
- Improves prediction of minority class

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

In [3]:
df = sns.load_dataset("titanic")

In [7]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [12]:
# Numeric columns
df["age"] = df["age"].fillna(df["age"].median())
df["fare"] = df["fare"].fillna(df["fare"].median())

# Categorical columns
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])
df["deck"] = df["deck"].fillna(df["deck"].mode()[0])

In [15]:
from imblearn.over_sampling import SMOTE 
from collections import Counter 

# Separate features and target
X = df.drop(columns=["survived", "alive"])
Y = df["survived"]

X = pd.get_dummies(X, drop_first=True)

# Display class distribution before SMOTE
print("Before SMOTE:")
print(Counter(Y))

# Apply SMOTE to balance the dataset
smote = SMOTE(random_state=42)

X_resampled, Y_resampled = smote.fit_resample(X,Y)

# Display class distribution after SMOTE
print("\nAfter SMOTE:")
print(Counter(Y_resampled))

Before SMOTE:
Counter({0: 549, 1: 342})

After SMOTE:
Counter({0: 549, 1: 549})


## 1.2 Random Undersampling

### What is Undersampling?

Random Undersampling reduces the number of samples from the majority class.

### Advantages

- Faster training
- Balanced dataset

### Disadvantage

Some useful information may be lost because majority samples are removed.

In [18]:
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter 

# Separate features and target
X = df.drop(columns=["survived", "alive"])
Y = df["survived"]

X = pd.get_dummies(X, drop_first=True)

# Display class distribution before undersampling
print("Before Undersampling:")
print(Counter(Y))

# Apply Random Undersampling 
under_sampler = RandomUnderSampler(random_state=42)

X_under, Y_under = under_sampler.fit_resample(X,Y)

# Display class distribution after SMOTE
print("\nAfter Undersampling:")
print(Counter(Y_under))

Before Undersampling:
Counter({0: 549, 1: 342})

After Undersampling:
Counter({0: 342, 1: 342})


## 1.3 Class Weights

### What are Class Weights?

Some Machine Learning models allow assigning higher importance to the minority class during training.

Instead of changing the dataset, the algorithm automatically gives more attention to minority samples.

In [20]:
from sklearn.ensemble import RandomForestClassifier

# Separate features and target
X = df.drop(columns=["survived", "alive"])
Y = df["survived"]

X = pd.get_dummies(X, drop_first=True)

# Train Random Forest using balanced class weights

model = RandomForestClassifier(class_weight = "balanced", random_state=42)

model.fit(X,Y)

# Display training accuracy
print("Training Accuracy:")
print(model.score(X, Y))

Training Accuracy:
0.9820426487093153


# 2. Time Series Preprocessing

Time Series data contains observations recorded over time.

Examples include:

- Stock Prices
- Weather Data
- Sales Data
- Temperature Records

Unlike normal datasets, time series data depends on chronological order.

## 2.1 Resampling

### What is Resampling?

Resampling changes the frequency of time series data.

Examples:

- Hourly → Daily
- Daily → Monthly
- Monthly → Yearly

In [ ]:
import pandas as pd

# Convert the date column into datetime format
df["date"] = pd.to_datetime(df["date"],errors="coerce")

# Set date as index
df.set_index("date",inplace=True)

# Resample data to monthly frequency
monthly_data = df.resample("M").mean(numeric_only=True)

print(monthly_data.head())

## 2.2 Lag Features

### What are Lag Features?

Lag features use previous observations as input features.

Example:

Today's value ← Yesterday's value

They are commonly used for forecasting.

In [22]:
# Create a lag feature using the previous row
df["lag_1"] = df["survived"].shift(1)

# Display the dataset
print(df.head())

   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  lag_1  
0    man        True    C  Southampton    no  False    NaN  
1  woman       False    C    Cherbourg   yes  False    0.0  
2  woman       False    C  Southampton   yes   True    1.0  
3  woman       False    C  Southampton   yes  False    1.0  
4    man        True    C  Southampton    no   True    1.0  


## 2.3 Rolling Window Features

### What are Rolling Features?

Rolling statistics calculate values over a moving window.

Common examples include:

- Moving Average
- Moving Sum
- Moving Standard Deviation

These features help capture trends over time.

In [23]:
# Calculate moving average using a window size of 3

df["rolling_mean"] = (df["survived"].rolling(window=3).mean())

print(df.head())

   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  lag_1  rolling_mean  
0    man        True    C  Southampton    no  False    NaN           NaN  
1  woman       False    C    Cherbourg   yes  False    0.0           NaN  
2  woman       False    C  Southampton   yes   True    1.0      0.666667  
3  woman       False    C  Southampton   yes  False    1.0      1.000000  
4    man        True    C  Southampton    no   True    1.0      0.666667  


-------
## 📝 Note

Advanced preprocessing techniques help Machine Learning models handle real-world data more effectively. Proper preprocessing improves model accuracy, robustness, and generalization.

---
## 📝 Important Note

Data preprocessing is one of the most important stages in the Machine Learning pipeline. A clean, well-structured, and properly transformed dataset helps build more accurate, reliable, and robust models.

➡️ In the next notebook, we will explore **Model Selection**, where different Machine Learning algorithms will be trained, compared, and evaluated to identify the best-performing model for the given dataset.